# JourneyGraph — LLM Pipeline Demo

**Prerequisites**: `.env` at the project root with `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` set. Neo4j running.

**Install**: `uv sync --extra demo`

---

This notebook walks through the full natural-language query pipeline end-to-end:

| Step | Component | What it does |
|---|---|---|
| 1 | **Planner** | Classifies domain, selects execution path, extracts anchor entities |
| 2 | **Anchor Resolver** | Maps entity strings to Neo4j node IDs via full-text index |
| 3a | **Subgraph Builder** | Hop-expands anchor nodes into a contextual subgraph |
| 3b | **Text-to-Cypher** | Generates and validates a Cypher query; up to 3 self-correcting attempts |
| 4 | **Narration Agent** | Produces the final plain-English answer from query results or graph context |
| + | **Agentic Pipeline** | Same question via dynamic tool-calling loop (Claude function calls) |

---
## Setup

Initialises the shared pipeline components used across all questions below.
The `SliceRegistry` validates YAML domain slices against the live graph before any LLM call is made.

In [14]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_root = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists()), None
)
if _root is None:
    raise RuntimeError("Could not find project root — no pyproject.toml in CWD or any parent.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.common.config import get_llm_config
from src.common.neo4j_tools import Neo4jManager
from src.llm.narration_agent import NarrationAgent
from src.llm.planner import Planner
from src.llm.slice_registry import SliceRegistry
from demos.pipeline_demo import (
    display_model_info,
    run_and_display,
    # Q1 step-by-step helpers
    display_question_header,
    display_planner_step,
    display_anchor_step,
    display_subgraph_step,
    display_t2c_step,
    display_narration_step,
    display_agentic_step,
    display_comparison_todo,
    run_question,
    visualize_subgraph,
)

llm_config = get_llm_config()
db = Neo4jManager()
registry = SliceRegistry(db, strict=False)
planner = Planner(registry, llm_config, strict=False)
narration_agent = NarrationAgent(llm_config)

print("Neo4j connected")
print(f"SliceRegistry domains: {registry.domains()}")
print("Planner and NarrationAgent ready")

Neo4j connected
SliceRegistry domains: ['accessibility', 'delay_propagation', 'transfer_impact']
Planner and NarrationAgent ready


---
## Model & Configuration

In [15]:
display_model_info(llm_config)

╔════════════════════════════════════════════════════════╗
║  Model       : claude-haiku-4-5-20251001                ║
║  Provider    : anthropic                                ║
║  Token budget: 512 (pipeline) / 1024 (narration)        ║
╚════════════════════════════════════════════════════════╝


---
## Questions

Three sample questions covering all three pipeline domains and both execution paths.

In [16]:
USER_QUERIES = [
    # Q1 — transfer_impact domain, GDS betweenness (network analysis)
    "Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?",

    # Q2 — delay_propagation domain, both paths (counts + topology)
    "Which bus trips have been delayed most recently and how severe were the delays?",

    # Q3 — accessibility domain, text2cypher (live equipment status)
    "Is the elevator at Metro Center currently out of service?",
]

for i, q in enumerate(USER_QUERIES, 1):
    print(f"  Q{i}: {q}")

  Q1: Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?
  Q2: Which bus trips have been delayed most recently and how severe were the delays?
  Q3: Is the elevator at Metro Center currently out of service?


---
---
# Question 1 — Full Walkthrough

> *"Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?"*

This section shows every pipeline stage in detail. Questions 2 and 3 use the same
stages internally but surface only the outputs.

This question exercises the **GDS (Graph Data Science)** path: the Planner detects
that betweenness centrality is required and sets `use_gds=True`, causing the
QueryWriter to inject GDS-specific few-shot examples and generate a named-graph
projection + algorithm stream query.

---
## Q1 · Step 1: Planner

A single LLM call performs three sub-tasks simultaneously:
- **Domain classification** — which knowledge domain applies (`transfer_impact`, `delay_propagation`, `accessibility`)
- **Path routing** — `text2cypher` for precise counts/lookups, `subgraph` for topology, `both` when needed
- **Anchor extraction** — station names, route names, dates pulled from the query

JSON parse failure triggers one automatic retry with a corrective nudge, then degrades to `text2cypher` with empty anchors.

In [17]:
q1 = USER_QUERIES[0]
print(f"Query: {q1!r}")

q1_planner_output = planner.run(q1)
display_planner_step(q1_planner_output)

Query: 'Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?'
── Step 1 · Planner ─────────────────────────────────────
  domain         : 'delay_propagation'
  path           : 'subgraph'
  path_reasoning : 'This query asks for structural network analysis to identify critical stations by their topological impact on route connectivity — a graph-theoretic centrality problem best answered via GDS betweenness or articulation-point detection.'
  anchor_notes   : None
  use_gds        : True
  anchors:
    routes         : []
    stations       : []
    dates          : []
    pathway_nodes  : []
    levels         : []


---
## Q1 · Step 2: Anchor Resolution

Maps the Planner's anchor strings to Neo4j node IDs:
- **Phase 1** — Full-text index lookup, generates candidate nodes per mention
- **Phase 2** — Disambiguation (`TopK` by default; `TypeWeightedCoherence` when `candidate_limit > 1`)
- Relative dates (`"yesterday"`, `"last Tuesday"`) are normalised to `YYYYMMDD`

In [18]:
from datetime import UTC, datetime
from src.llm.anchor_resolver import AnchorResolver

q1_invocation_time = datetime.now(UTC)
q1_resolver = AnchorResolver(db=db, invocation_time=q1_invocation_time)
q1_resolutions = q1_resolver.resolve(q1_planner_output.anchors)

display_anchor_step(q1_resolutions)

anchor_resolver | zero candidates generated | anchors=PlannerAnchors(stations=[], routes=[], dates=[], pathway_nodes=[], levels=[])



── Step 2 · Anchor Resolution ───────────────────────────
  (no anchors resolved)


---
## Q1 · Step 3a: Subgraph Builder

Runs only when the Planner selects `subgraph` or `both`.

- **HopExpander** — bidirectional hop expansion from anchor nodes, constrained by domain-specific `DomainExpansionConfig`
- **ContextSerializer** — serialises to a text block capped at 2,000 tokens (`tiktoken cl100k_base`)
- The raw node+edge graph is rendered below for visualisation

In [19]:
from src.llm.hop_expander import HopExpander
from src.llm.subgraph_builder import SubgraphBuilder

q1_subgraph_output = None
q1_raw_subgraph = None

if q1_planner_output.path in {"subgraph", "both"}:
    expander = HopExpander(db=db)
    q1_raw_subgraph = expander.expand(q1_resolutions, q1_planner_output.domain)

    builder = SubgraphBuilder(db=db)
    q1_subgraph_output = builder.run(
        q1_planner_output, q1_resolutions, resolver_config=q1_resolver.config
    )

display_subgraph_step(q1_subgraph_output)

hop_expander | no anchor nodes seeded | domain=delay_propagation
subgraph_builder | zero anchors in resolutions (unexpected) | domain=delay_propagation



── Step 3a · Subgraph ───────────────────────────────────
  success    : False
  failure    : No anchors resolved from query


In [20]:
# Graph visualisation — shown when subgraph path was taken
if q1_raw_subgraph and q1_raw_subgraph.nodes:
    visualize_subgraph(q1_raw_subgraph, title="Q1 Subgraph")
else:
    print("Subgraph path not taken for Q1 — visualisation skipped.")
    print(f"(Planner selected path={q1_planner_output.path!r})")

Subgraph path not taken for Q1 — visualisation skipped.
(Planner selected path='subgraph')


---
## Q1 · Step 3b: Text-to-Cypher

Runs only when the Planner selects `text2cypher` or `both`.

**QueryWriter** (up to 3 self-correcting attempts):
1. Loads `conventions.json` and domain-specific `.cypher` few-shot examples
2. Builds a system prompt: role + node/relationship whitelist + WMATA data quirks + few-shot examples
3. Makes a single LLM call → returns a ```` ```cypher ``` ```` block + chain-of-thought explanation

**CypherValidator** (per attempt): write-clause guard → blocked-CALL guard → `EXPLAIN` syntax check → label/rel/property whitelist. Validation errors are fed back as refinement hints on retry.

In [21]:
from src.common.logger import get_logger
from src.llm.cypher_validator import validate_and_log_cypher
from src.llm.query_writer import run_query_writer
from src.llm.text2cypher_output import Text2CypherOutput

_log = get_logger(__name__)
_MAX_ATTEMPTS = 3

q1_t2c_output = None
q1_t2c_cot = None

if q1_planner_output.path in {"text2cypher", "both"}:
    schema_slice = registry.get(q1_planner_output.schema_slice_key)
    refinement_errors: list[str] = []
    all_validation_notes: list[str] = []

    for attempt in range(1, _MAX_ATTEMPTS + 1):
        qw_output = run_query_writer(
            q1,
            q1_planner_output,
            llm_config,
            schema_slice=schema_slice,
            resolved_anchors=q1_resolutions.as_flat_dict(),
            refinement_errors=refinement_errors or None,
            use_gds=q1_planner_output.use_gds,
        )
        if q1_t2c_cot is None:
            q1_t2c_cot = qw_output.cot_comments

        print(f"[QueryWriter — attempt {attempt}/{_MAX_ATTEMPTS}]")
        print("Cypher:")
        print(qw_output.cypher_query)
        print("\nChain-of-Thought:")
        print(qw_output.cot_comments)

        val_result = validate_and_log_cypher(
            qw_output.cypher_query,
            schema_slice,
            schema_slice.property_registry,
            db.driver,
            _log,
        )
        if val_result.valid:
            print(f"\nValidator: ✓ valid (attempt {attempt}) — results: {val_result.results}")
            q1_t2c_output = Text2CypherOutput(
                cypher=qw_output.cypher_query,
                results=val_result.results or [],
                domain=q1_planner_output.domain,
                attempt_count=attempt,
                validation_notes=all_validation_notes,
                success=True,
            )
            break
        print(f"\nValidator: ✗ invalid — {val_result.errors}")
        all_validation_notes.extend(val_result.errors)
        refinement_errors = val_result.errors
    else:
        print(f"All {_MAX_ATTEMPTS} attempts failed.")
        q1_t2c_output = Text2CypherOutput(
            cypher="",
            results=[],
            domain=q1_planner_output.domain,
            attempt_count=_MAX_ATTEMPTS,
            validation_notes=all_validation_notes,
            success=False,
        )
else:
    print(f"Text2Cypher path not taken (path={q1_planner_output.path!r})")

Text2Cypher path not taken (path='subgraph')


---
## Q1 · Step 4: Narration Agent (Static Pipeline)

Terminal LLM call. Response mode is chosen by pure Python logic — no LLM involved:

| Mode | Condition |
|---|---|
| `precision` | Text2Cypher succeeded, no subgraph |
| `contextual` | Subgraph succeeded, no Text2Cypher |
| `synthesis` | Both paths succeeded |
| `degraded` | Neither path succeeded |

In [22]:
q1_narration_output = narration_agent.run(
    q1,
    q1_planner_output,
    t2c_output=q1_t2c_output,
    subgraph_output=q1_subgraph_output,
    resolutions=q1_resolutions,
)

display_narration_step(q1_narration_output, label="Static")


── Step 4 · Narration [Static] ──────────────────────────
  mode    : degraded
  sources : []

  Answer:
════════════════════════════════════════════════════════
I cannot answer this question with the data currently available to me.

**What was not resolved:**
- No specific stations, routes, or dates were provided in your query
- No graph data about network topology or route connectivity is present in my current context
- No operational data about station criticality or route dependencies is available

**What you're asking for:**
You're asking for a network vulnerability analysis — identifying "chokepoints" or critical stations whose closure would affect the most routes. This requires:
1. Complete network topology data (which stations serve which routes)
2. Route connectivity information (how routes overlap at stations)
3. Alternative routing paths (to determine if closure leaves routes with no alternative)

**To help you, please provide:**
- Specific stations you're concerned about, 

---
## Q1 · Agentic Pipeline

Runs `AgentOrchestrator` on the same question using the same Planner output and anchor resolutions.

Instead of the fixed QueryWriter → Validator → Subgraph fork, a **Claude function-calling loop** (max 5 iterations) selects tools dynamically:

| Tool | Purpose |
|---|---|
| `cypher_query` | QueryWriter + CypherValidator |
| `subgraph_expand` | SubgraphBuilder hop expansion |
| `full_text_search` | Additional entity lookup via AnchorResolver |
| `entity_clarify` | AnchorClarifier repair pass |

In [23]:
from src.llm.agent import AgentOrchestrator
from src.llm.anchor_clarifier import AnchorClarifier

q1_clarifier = AnchorClarifier(db, llm_config)
q1_orchestrator = AgentOrchestrator(
    db=db,
    llm_config=llm_config,
    registry=registry,
    clarifier=q1_clarifier,
    narration_agent=narration_agent,
)

_, _, q1_agent_narration = q1_orchestrator.run(
    q1, q1_planner_output, q1_resolutions, q1_resolver, q1_invocation_time
)

display_agentic_step(q1_agent_narration)


── Agentic Pipeline ─────────────────────────────────────
  mode    : synthesis
  sources : ['text2cypher', 'subgraph']
  tools   : 2 call(s)
    [?] ? — 
    [?] ? — 

  Answer:
════════════════════════════════════════════════════════
# Critical Chokepoints: L'Enfant Plaza and Metro Center

Based on network connectivity analysis, **two stations stand out as critical chokepoints**:

## Primary Chokepoint: L'Enfant Plaza
- **Routes affected if closed: 5**
- **Station ID: STN_D03_F03**

L'Enfant Plaza is the single most critical station on the WMATA network. It is a major transfer hub where **five routes converge**, making it indispensable for network connectivity. A closure here would sever service across the largest number of routes simultaneously.

## Secondary Chokepoint: Metro Center
- **Routes affected if closed: 4**
- **Station ID: STN_A01_C01**

Metro Center is the second-highest-criticality station, serving **four routes**. As a downtown hub, its closure would significantly fra

---
## Q1 · Comparison Notes

**Static pipeline**
- Mode: `precision` — Text2Cypher succeeded, no subgraph (GDS path selected text2cypher only)
- Sources: `text2cypher`
- Answer: Tiered betweenness breakdown — Metro Center #1 (1263.18), L'Enfant Plaza #2 (943.93), Fort Totten / Gallery Place tied at 350.28, then a tail of terminal stations at 27.7. Clear prose answer with a "key finding" callout.

**Agentic pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Tools: 1 call
- Answer: Answered from degree (route count) rather than betweenness — L'Enfant Plaza #1 (5 routes), Metro Center #2 (4 routes), 13 stations tied at 3. Different algorithm, same story: same two stations dominate.

**Observations**
- Both pipelines converge on Metro Center and L'Enfant Plaza as the top two nodes, but via different metrics (betweenness vs. degree) — a good illustration that the agentic loop chose its own retrieval strategy
- Static answer is more analytically precise (uses the correct betweenness metric implied by "choke point")
- GDS named-graph projection generated and executed correctly on the first attempt — no retry needed


---
---
# Question 2

> *"Which bus trips have been delayed most recently and how severe were the delays?"*

Same pipeline stages as Q1 — see the Q1 walkthrough above for implementation details.

This question uses the **`both` path** — Text2Cypher retrieves exact delay counts and
severity per route, while the Subgraph Builder expands the network topology around the
affected trips. The NarrationAgent synthesises both sources into a single answer.

No temporal anchor is needed — "most recently" resolves to the latest Date node in
the graph automatically.

In [24]:
result2 = run_and_display(
    USER_QUERIES[1], 2, db, llm_config, registry, planner, narration_agent
)

anchor_resolver | route not found | name=bus
anchor_resolver | date unresolvable | expr=most recently
anchor_resolver | zero candidates generated | anchors=PlannerAnchors(stations=[], routes=['bus'], dates=['most recently'], pathway_nodes=[], levels=[])
hop_expander | no anchor nodes seeded | domain=delay_propagation
subgraph_builder | zero anchors in resolutions (unexpected) | domain=delay_propagation



════════════════════════════════════════════════════════
  Q2: Which bus trips have been delayed most recently and how severe were the delays?
════════════════════════════════════════════════════════

── Step 1 · Planner ─────────────────────────────────────
  domain         : 'delay_propagation'
  path           : 'both'
  path_reasoning : 'The query requires both a precise ranking/count of delayed bus trips (text2cypher) and severity context that may benefit from explanatory detail (subgraph).'
  anchor_notes   : "Route anchored as generic 'bus' since no specific route number was provided; resolver will interpret 'most recently' as the latest available disruption data."
  use_gds        : False
  anchors:
    routes         : ['bus']
    stations       : []
    dates          : ['most recently']
    pathway_nodes  : []
    levels         : []

── Step 2 · Anchor Resolution ───────────────────────────
  (no anchors resolved)
  failed: {'bus': "No Route matched 'bus'", 'most recently'

---
## Q2 · Comparison Notes

**Static pipeline**
- Mode: `precision` — Text2Cypher returned results; Subgraph failed (anchor "bus" unresolvable, no station seed)
- Sources: `text2cypher`
- Answer: Top 20 delayed bus trips on 2026-04-14. 3 SEVERE delays (C15: 19.2 min, M44: 15 min, D70: 15 min), 17 WARNING. QueryWriter correctly handled the no-anchor case by using the two-pass temporal pattern (`ORDER BY d.date DESC LIMIT 1`) to resolve "most recently" without a date node.

**Agentic pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Tools: 1 call
- Answer: Same 20 trips, ranked by delay duration with a cleaner table. Surfaced an additional SEVERE trip (C25: 941s) not highlighted by the static answer.

**Observations**
- Anchor resolution correctly failed on "bus" and "recently" — neither maps to a graph node — but Text2Cypher recovered gracefully by writing a no-anchor temporal query
- Subgraph path was blocked by missing anchors; precision mode still produced a useful answer
- Both pipelines produced identical data, but the agentic answer ranks by duration rather than recency — slightly more useful for understanding severity
- The raw `started` timestamps appear as Unix epoch in both answers — a narration prompt improvement opportunity


---
---
# Question 3

> *"Is the elevator at Metro Center currently out of service?"*

Same pipeline stages as Q1 — see the Q1 walkthrough above for implementation details.

This question exercises the **accessibility domain**. An active `OutageEvent` node
exists in the graph for the Metro Center street-to-mezzanine elevator (unit `A01L01`,
symptom: Door Malfunction), so the pipeline should return a concrete outage answer
rather than an all-clear.

In [25]:
result3 = run_and_display(
    USER_QUERIES[2], 3, db, llm_config, registry, planner, narration_agent
)


════════════════════════════════════════════════════════
  Q3: Is the elevator at Metro Center currently out of service?
════════════════════════════════════════════════════════

── Step 1 · Planner ─────────────────────────────────────
  domain         : 'accessibility'
  path           : 'text2cypher'
  path_reasoning : 'This is a binary yes/no lookup for current elevator status at a specific station, requiring a direct Cypher query against the accessibility status graph.'
  anchor_notes   : "The query references 'the elevator' generically; the station anchor Metro Center will resolve to its associated elevator assets."
  use_gds        : False
  anchors:
    routes         : []
    stations       : ['Metro Center']
    dates          : ['today']
    pathway_nodes  : []
    levels         : []

── Step 2 · Anchor Resolution ───────────────────────────
  'Metro Center'                 → ['STN_A01_C01']
  'today'                        → ['20260414']

── Step 3a · Subgraph ───────────

---
## Q3 · Comparison Notes

**Static pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Answer: Correctly identified one elevator out of service (A01_C01_104115, street to mezzanine, Door Malfunction) and two operational. Note: narration reported `outage_status: resolved` verbatim from the fake node — the ETA timestamp also surfaced as raw epoch.

**Agentic pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Tools: 1 call
- Answer: Same outage identified, presented as a cleaner table. Attempted to humanise the ETA timestamp but computed it incorrectly (showed March 2025 instead of April 2026 — likely a unit conversion error on the epoch value).

**Observations**
- The fake OutageEvent node worked — the pipeline found it via `OPTIONAL MATCH (o:OutageEvent)-[:AFFECTS]->(e)` and correctly split the three elevators into one outage + two operational
- Partial outage (1 of 3) is more interesting than an all-clear or total failure
- Two narration issues to note: (1) raw epoch ETA leaks through in both pipelines, (2) `outage_status: resolved` from the fake node is misleading — the fake node's `status` field should be `"active"` not `"resolved"` to avoid this


---
## Teardown

In [26]:
db.close()
print("Neo4j connection closed.")

Neo4j connection closed.
